## Self-Training of the Regressions

In this part we use the rows where a target is missing to train the regressions of `yield_strength_MPa`, `uts_MPa` and `elongation_pct`. The Extra-Trees predict the missing values, the most confident predictions become pseudo-labels, and the model is trained again with them.

### Main observations

- **Pool:** 192 rows for the yield strength, 217 for the UTS and 251 for the elongation, against 558 to 620 labelled rows. The other rows with a missing target copy the inputs of a labelled row.
- **Pool against labelled rows:** A random forest tells them apart with an AUC of 0.990 to 0.996. A pool row is at a median distance of 1.41 to 3.97 from the nearest labelled row, against 0.89 to 0.91 between labelled rows of different folds. The pseudo-labels are extrapolations.
- **Confidence:** On the labelled rows, the spread of the trees ranks the errors: 18.9 MPa of MAE in the fifth with the lowest spread against 48.1 MPa in the highest, for the yield strength. The Spearman correlation is only 0.23 to 0.36.
- **Self-training:** No gain. The MAE goes from 30.35 to 31.09 MPa for the yield strength, from 26.92 to 26.75 MPa for the UTS, and from 1.835 to 1.841% for the elongation. Every change per fold is below 2 MPa, against a standard deviation of about 3 MPa over the folds.
- **Label:** F1 of 0.797 without pseudo-labels and 0.788 with them. 3 of the 175 labelled rows change label.
- **Conclusion:** The pool rows are too far from the labelled rows for the pseudo-labels to add information.

### Operations in order

1. Build the pool of each target: the rows where the target is missing, one per distinct input, without the inputs of a labelled row.
2. Compare the pool with the labelled rows: a random forest tells them apart, and the distance to the nearest labelled row.
3. Check the confidence: out of fold on the labelled rows, the MAE per fifth of the spread of the trees.
4. Self-train on the folds of `helpers/utils.py`: 3 rounds, each adds the 25% of the pool with the lowest spread, and compare with the same model without pseudo-labels.
5. Apply the thresholds to both and compare the label.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append("..")  # helpers/ is in modeling/

from helpers.utils import (
    SEED, N_FOLDS, TARGETS, CHARPY, INPUT_COLS,
    load_train, get_xy, build_pipeline, oof_predict, true_criteria, predicted_criteria, deposit_labels,
)

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

In [3]:
TENSILE = TARGETS[:3]
SHARE = 0.25   # share of the pool added per round
ROUNDS = 3


def make_model(max_features=0.5):
    """Extra-Trees of 1B4, with bootstrap so that the trees disagree on the training rows too."""
    return ExtraTreesRegressor(n_estimators=200, max_features=max_features, bootstrap=True, random_state=SEED)


def mean_and_spread(pipe, X):
    """Mean and standard deviation of the predictions of the trees of a fitted pipeline."""
    X_scaled = pipe[:-1].transform(X)
    trees = np.stack([tree.predict(X_scaled) for tree in pipe[-1].estimators_])
    return trees.mean(axis=0), trees.std(axis=0)

### Pool

In [4]:
def build_pool(df, target):
    """Rows where `target` is missing, one per distinct input, without the inputs of a labelled row.

    `known_group` is True when another row of the same input_group has `target`, so the row is the
    same weld with another PWHT.
    """
    known = df[target].notna()
    labelled_inputs = set(map(tuple, df.loc[known, INPUT_COLS].to_numpy()))
    rows = df.loc[~known].drop_duplicates(INPUT_COLS)
    rows = rows[[tuple(x) not in labelled_inputs for x in rows[INPUT_COLS].to_numpy()]].copy()
    rows["known_group"] = rows["input_group"].isin(df.loc[known, "input_group"])
    return rows


train = load_train()
pools = {target: build_pool(train, target) for target in TENSILE}

pd.DataFrame({
    target: {
        "labelled_rows": train[target].notna().sum(),
        "missing_rows": train[target].isna().sum(),
        "pool_rows": len(pool),
        "pool_rows_known_group": pool["known_group"].sum(),
        "pool_rows_new_group": (~pool["known_group"]).sum(),
        **{f"pool_rows_fold_{k}": (pool["fold"] == k).sum() for k in range(N_FOLDS)},
    }
    for target, pool in pools.items()
}).T

,labelled_rows,missing_rows,pool_rows,pool_rows_known_group,pool_rows_new_group,pool_rows_fold_0,pool_rows_fold_1,pool_rows_fold_2,pool_rows_fold_3,pool_rows_fold_4
yield_strength_MPa,620,712,192,127,65,39,37,42,39,35
uts_MPa,596,736,217,127,90,46,40,49,46,36
elongation_pct,558,774,251,128,123,59,46,54,50,42


- **Pool:** 192 rows for the yield strength, 217 for the UTS and 251 for the elongation, against 558 to 620 labelled rows. For the yield strength, the other 520 of the 712 rows without a value copy the inputs of a labelled row or of another pool row, so they add nothing.
- **Known group:** 127 or 128 pool rows per target belong to an `input_group` where the target is measured on another row. They are the same weld with another PWHT. The other 65 to 123 pool rows are new welds.
- **Folds:** 35 to 59 pool rows per fold.

A random forest learns to tell a labelled row from a pool row, from the inputs only, on the same folds. The distance is the Euclidean distance on the standardised inputs: for a labelled row, to the nearest labelled row of the other folds, as seen by its out-of-fold prediction; for a pool row, to the nearest labelled row.

In [5]:
summary = {}
for target in TENSILE:
    X, y, _ = get_xy(train, target)
    pool = pools[target]
    fold = train.loc[X.index, "fold"].to_numpy()

    inputs = np.vstack([X.to_numpy(), pool[INPUT_COLS].to_numpy()])
    is_pool = np.r_[np.zeros(len(X)), np.ones(len(pool))]
    folds = np.r_[fold, pool["fold"].to_numpy()]
    score = np.zeros(len(inputs))
    for k in range(N_FOLDS):
        forest = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(inputs[folds != k], is_pool[folds != k])
        score[folds == k] = forest.predict_proba(inputs[folds == k])[:, 1]

    scaler = StandardScaler().fit(X)
    X_scaled, pool_scaled = scaler.transform(X), scaler.transform(pool[INPUT_COLS])
    labelled_distance = np.empty(len(X))
    for k in range(N_FOLDS):
        nearest = NearestNeighbors(n_neighbors=1).fit(X_scaled[fold != k])
        labelled_distance[fold == k] = nearest.kneighbors(X_scaled[fold == k])[0][:, 0]
    pool_distance = NearestNeighbors(n_neighbors=1).fit(X_scaled).kneighbors(pool_scaled)[0][:, 0]
    known_group = pool["known_group"].to_numpy()

    summary[target] = {
        "AUC_labelled_vs_pool": roc_auc_score(is_pool, score),
        "median_distance_labelled_rows": np.median(labelled_distance),
        "median_distance_pool_known_group": np.median(pool_distance[known_group]),
        "median_distance_pool_new_group": np.median(pool_distance[~known_group]),
    }

pd.DataFrame(summary).T.round(3)

,AUC_labelled_vs_pool,median_distance_labelled_rows,median_distance_pool_known_group,median_distance_pool_new_group
yield_strength_MPa,0.996,0.905,1.412,3.967
uts_MPa,0.995,0.888,1.405,2.829
elongation_pct,0.990,0.888,1.427,2.907


- **AUC:** 0.990 to 0.996. The forest tells a pool row from a labelled row almost every time.
- **Distance:** A labelled row is at a median distance of 0.89 to 0.91 from the nearest labelled row of the other folds. A pool row of a known group is at 1.41 to 1.43, a pool row of a new group at 2.83 to 3.97. The pool rows are farther from the labelled rows than the labelled rows are from each other, so their predictions are extrapolations.

### Confidence

In [6]:
def oof_mean_and_spread(df, target):
    """Out-of-fold mean and spread of the trees on the labelled rows of `target`, and the absolute error."""
    X, y, cv = get_xy(df, target)
    out = pd.DataFrame(np.nan, index=X.index, columns=["pred", "spread"])
    for train_idx, val_idx in cv.split():
        pipe = build_pipeline(make_model()).fit(X.iloc[train_idx], y.iloc[train_idx])
        mean, spread = mean_and_spread(pipe, X.iloc[val_idx])
        out.iloc[val_idx, 0] = mean
        out.iloc[val_idx, 1] = spread
    out["abs_error"] = (out["pred"] - y).abs()
    return out


oof = {target: oof_mean_and_spread(train, target) for target in TENSILE}
fifths = ["1, lowest spread", "2", "3", "4", "5, highest spread"]

confidence = pd.DataFrame({
    target: out.groupby(pd.qcut(out["spread"], 5, labels=fifths), observed=True)["abs_error"].mean()
    for target, out in oof.items()
})
confidence.loc["all labelled rows"] = [out["abs_error"].mean() for out in oof.values()]
confidence.loc["Spearman, spread against error"] = [spearmanr(out["spread"], out["abs_error"]).statistic for out in oof.values()]
confidence.round(3)

,yield_strength_MPa,uts_MPa,elongation_pct
spread,,,
"1, lowest spread",18.891,15.485,1.325
2,22.377,19.054,1.559
3,31.429,28.037,1.697
4,30.952,26.657,2.026
"5, highest spread",48.077,45.442,2.566
all labelled rows,30.345,26.916,1.835
"Spearman, spread against error",0.314,0.358,0.233


The spread is the standard deviation of the predictions of the 200 trees. The labelled rows are split into 5 groups of the same size by spread.

- **Fifths:** The MAE rises with the spread: 18.9 MPa in the fifth with the lowest spread against 48.1 MPa in the highest for the yield strength, 15.5 against 45.4 MPa for the UTS, 1.33 against 2.57% for the elongation. Fifths 3 and 4 are close for the yield strength and the UTS.
- **Spearman:** 0.23 to 0.36. The spread ranks the errors, but loosely.
- **Verdict:** On the labelled rows, a low spread means a lower error. The pool rows are farther from the training rows, so this is not shown for them.

### Self-training

In [7]:
def self_training(df, target, pool, k):
    """Predictions of the labelled rows of fold k without and with pseudo-labels, and the log of the rounds.

    The model is trained on the labelled rows of the 4 other folds. Each round predicts the pool rows of
    the 4 other folds not yet added, adds the 25% of the initial pool with the lowest spread, with the
    mean of the trees as pseudo-label, and trains the model again.
    """
    X, y, _ = get_xy(df, target)
    fold = df.loc[X.index, "fold"].to_numpy()
    X_train, y_train, X_val = X[fold != k], y[fold != k], X[fold == k]
    remaining = pool.loc[pool["fold"] != k, INPUT_COLS]
    n_add = int(np.ceil(SHARE * len(remaining)))

    pipe = build_pipeline(make_model()).fit(X_train, y_train)
    reference = pd.Series(pipe.predict(X_val), index=X_val.index)

    added_X, added_y, log = [], [], []
    for r in range(1, ROUNDS + 1):
        mean, spread = mean_and_spread(pipe, remaining)
        best = np.argsort(spread)[:n_add]
        added_X.append(remaining.iloc[best])
        added_y.append(pd.Series(mean[best], index=remaining.index[best]))
        log.append({"round": r, "added": len(best), "max_spread_added": spread[best].max(), "max_spread_pool": spread.max()})
        remaining = remaining.drop(remaining.index[best])
        pipe = build_pipeline(make_model()).fit(pd.concat([X_train, *added_X]), pd.concat([y_train, *added_y]))

    return reference, pd.Series(pipe.predict(X_val), index=X_val.index), log


preds, logs = {}, []
for target in TENSILE:
    reference, self_trained = [], []
    for k in range(N_FOLDS):
        ref, st, log = self_training(train, target, pools[target], k)
        reference.append(ref)
        self_trained.append(st)
        logs += [{"target": target, "fold": k, **entry} for entry in log]
    preds[target] = {"reference": pd.concat(reference), "self_training": pd.concat(self_trained)}

pd.DataFrame(logs).groupby(["target", "round"], sort=False)[["added", "max_spread_added", "max_spread_pool"]].mean().round(3)

added  max_spread_added  max_spread_pool
target             round                                          
yield_strength_MPa 1       39.0            34.962          123.264
                   2       39.0            36.062          122.158
                   3       39.0            75.432          121.249
uts_MPa            1       43.8            30.821          125.749
                   2       43.8            35.048          127.103
                   3       43.8            61.842          122.726
elongation_pct     1       50.8             2.278            6.144
                   2       50.8             2.803            6.400
                   3       50.8             3.637            6.274

The mean over the 5 folds of each round.

- **Added rows:** 39 rows per round for the yield strength, 44 for the UTS and 51 for the elongation.
- **Spread:** The rows added in round 1 have a spread of at most 31 to 35 MPa for the tensile strengths, those of round 3 up to 62 to 75 MPa, against 121 to 127 MPa for the least confident pool rows. Each round adds less confident rows.

In [8]:
rows = []
for target, arms in preds.items():
    y = train.loc[arms["reference"].index, target]
    fold = train.loc[y.index, "fold"]
    errors = pd.DataFrame({arm: pred - y for arm, pred in arms.items()})
    per_fold = errors.abs().groupby(fold).mean()
    change = per_fold["self_training"] - per_fold["reference"]
    rows.append({
        "target": target,
        "MAE_reference": errors["reference"].abs().mean(),
        "MAE_self_training": errors["self_training"].abs().mean(),
        "RMSE_reference": np.sqrt((errors["reference"] ** 2).mean()),
        "RMSE_self_training": np.sqrt((errors["self_training"] ** 2).mean()),
        "folds_improved": int((change < 0).sum()),
        "fold_MAE_change_min": change.min(),
        "fold_MAE_change_max": change.max(),
    })

pd.DataFrame(rows).set_index("target").round(3)

,MAE_reference,MAE_self_training,RMSE_reference,RMSE_self_training,folds_improved,fold_MAE_change_min,fold_MAE_change_max
target,,,,,,,
yield_strength_MPa,30.345,31.090,43.544,44.382,1,-0.111,1.881
uts_MPa,26.916,26.745,40.017,40.074,3,-0.649,0.462
elongation_pct,1.835,1.841,2.405,2.410,3,-0.011,0.029


- **Reference:** MAE of 30.35 MPa, 26.92 MPa and 1.835%, the values of the Extra-Trees with bootstrap in `1B4`.
- **Yield strength:** 31.09 MPa with self-training, worse in 4 folds of 5.
- **UTS:** 26.75 MPa, better in 3 folds. **Elongation:** 1.841%, better in 3 folds.
- **Size:** The change per fold goes from -0.65 to +1.88 MPa, against a standard deviation of the MAE over the folds of 2.8 to 3.2 MPa for the Extra-Trees in `results.csv`. No change is larger than the noise.
- **Verdict:** Self-training does not improve the regressions.

### Label

In [9]:
charpy = oof_predict(make_model(max_features=1.0), train, CHARPY)
true_crit = true_criteria(train)

label = {}
for arm in ["reference", "self_training"]:
    arm_preds = {target: pd.DataFrame({"pred": preds[target][arm], "pred_ref": preds[target][arm]}) for target in TENSILE}
    arm_preds[CHARPY] = charpy
    label[arm] = deposit_labels(train, true_crit, predicted_criteria(train, arm_preds))

pd.DataFrame({
    arm: {
        "F1_nc": f1_score(lab["true_nc"], lab["pred_nc"]),
        "recall_nc": recall_score(lab["true_nc"], lab["pred_nc"]),
        "precision_good": precision_score(lab["true_nc"] == 0, lab["pred_nc"] == 0),
        "labelled_rows": len(lab),
    }
    for arm, lab in label.items()
}).T.round(3)

,F1_nc,recall_nc,precision_good,labelled_rows
reference,0.797,0.663,0.767,175.0
self_training,0.788,0.651,0.760,175.0


In [10]:
changed = label["reference"]["pred_nc"] != label["self_training"]["pred_nc"]
print("Rows whose predicted label changes:", changed.sum(), "of", len(changed))
label["reference"].loc[changed, ["family", "true_nc"]].assign(
    reference=label["reference"].loc[changed, "pred_nc"],
    self_training=label["self_training"].loc[changed, "pred_nc"],
)

Rows whose predicted label changes: 3 of 175


,family,true_nc,reference,self_training
input_group,,,,
7,C-Mn,1,1,0
401,C-Mn,1,0,1
517,C-Mn,1,1,0


- **Label:** F1 of 0.797 without pseudo-labels and 0.788 with them, recall of 0.663 and 0.651, precision on the good rows of 0.767 and 0.760.
- **Changed rows:** 3 of the 175 labelled rows change label, all of them bad. Self-training catches one more and misses two more.
- **Bootstrap:** The reference F1, 0.797, is below the 0.84 of the Extra-Trees of `1B4` without bootstrap. The bootstrap needed for the spread costs some F1 on the label.